<a href="https://colab.research.google.com/github/Civio-Creative/sample_outputs/blob/main/AI_Bootcamp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

# **Schema discipline:** every field is lifted from text or trivially derivable. No subjective classification (urgency, sentiment, intent) — those belong in a separate pass where they can be evaluated independently.

# %%
import os
import json
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Optional

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

# %%
class ExtractedEmail(BaseModel):
    sender_name: Optional[str] = Field(description="Full name of sender if identifiable, else null")
    sender_email: Optional[str] = Field(description="Email address of sender")
    sender_company: Optional[str] = Field(description="Company name if mentioned in signature or domain, else null")
    subject: str = Field(description="Subject line, verbatim")
    is_reply: bool = Field(description="True if subject starts with Re: or body contains quoted prior message")
    mentioned_dates: list[str] = Field(description="Dates mentioned in body, normalized to ISO 8601 (YYYY-MM-DD). Empty list if none.")
    mentioned_dollar_amounts: list[float] = Field(description="Dollar amounts mentioned as floats. Empty list if none.")
    questions_asked: list[str] = Field(description="Literal questions the sender asks, verbatim from the email. Empty list if none.")
    requested_actions: list[str] = Field(description="Concrete actions the sender is asking the recipient to take. Empty list if none.")


SYSTEM_PROMPT = """You extract structured data from sales emails. Follow the schema exactly.

Rules:
- Extract only what is verifiable from the text. Do not infer or guess.
- Normalize dates to ISO 8601 (YYYY-MM-DD). If a date is ambiguous (e.g., "next Tuesday"), skip it.
- Dollar amounts as floats without currency symbols.
- Questions must be verbatim from the email, ending with a question mark.
- If a field cannot be determined from the text, use null (for optional fields) or an empty list.

Example 1:
Input:
---
From: Sarah Chen <sarah@acmecorp.com>
Subject: Pricing question for Q1 rollout

Hi team,

We're evaluating vendors for a rollout starting 2025-03-15. Our budget is around $45,000.
Can you share pricing for 50 seats? Also, do you offer volume discounts?

Please send a proposal by end of week.

Thanks,
Sarah Chen
Acme Corp
---
Output:
{
  "sender_name": "Sarah Chen",
  "sender_email": "sarah@acmecorp.com",
  "sender_company": "Acme Corp",
  "subject": "Pricing question for Q1 rollout",
  "is_reply": false,
  "mentioned_dates": ["2025-03-15"],
  "mentioned_dollar_amounts": [45000.00],
  "questions_asked": ["Can you share pricing for 50 seats?", "Also, do you offer volume discounts?"],
  "requested_actions": ["Send a proposal by end of week"]
}

Example 2 (missing data, forwarded):
Input:
---
From: unknown@gmail.com
Subject: Re: Fwd: quick question

> Original message: what's the deal
saw ur website. call me
---
Output:
{
  "sender_name": null,
  "sender_email": "unknown@gmail.com",
  "sender_company": null,
  "subject": "Re: Fwd: quick question",
  "is_reply": true,
  "mentioned_dates": [],
  "mentioned_dollar_amounts": [],
  "questions_asked": [],
  "requested_actions": ["Call the sender"]
}
"""

def extract_email(email_text: str) -> ExtractedEmail:
    response = client.responses.parse(
        model="gpt-4o-2024-08-06",
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Extract from this email:\n---\n{email_text}\n---"},
        ],
        text_format=ExtractedEmail,
    )
    return response.output_parsed



# %%
test_emails = [
    # Clean case
    """From: Marcus Rivera <m.rivera@northstar.io>
Subject: Interested in your enterprise plan

Hi,

I'm the VP of Ops at Northstar Logistics. We're looking to onboard around 200 users starting April 1, 2025.
Our budget is $85,000 for year one.

What's included in the enterprise tier? Can we schedule a demo next week?

Best,
Marcus""",

    # Forwarded chain, ambiguous
    """From: jenny@example.com
Subject: Fwd: Re: Fwd: pricing

---------- Forwarded message ----------
From: someone else
> we need to know cost for 30 seats and 100 seats
> also does it integrate with salesforce

can you get back to me by 12/15?""",

    # Signature-only company, multiple amounts, no clear ask
    """From: David Okafor <d.okafor@meridian-health.org>
Subject: Following up

Circling back on our conversation. We're comparing three vendors — your $12,500 quote, another at $9,800,
and a third at $15,000.

--
David Okafor
Director of IT | Meridian Health
Sent from my iPhone""",

    # Reply, embedded quote, one clear question
    """From: priya.s@bluetechlabs.com
Subject: Re: Contract terms

Thanks for sending this over.

> Section 4.2 covers termination with 30 days notice

Can we change that to 60 days notice? Everything else looks fine to sign by 2025-02-28.

Priya""",
]

# %%
results = []
for i, email in enumerate(test_emails, 1):
    print(f"\n{'='*60}\nEmail {i}\n{'='*60}")
    result = extract_email(email)
    print(json.dumps(result.model_dump(), indent=2))
    results.append(result)


# %%
def check_consistency(email_text: str, n: int = 3) -> dict:
    runs = [extract_email(email_text).model_dump() for _ in range(n)]
    consistent = all(runs[0] == r for r in runs[1:])
    return {"consistent": consistent, "runs": runs}

# %%
for i, email in enumerate(test_emails, 1):
    check = check_consistency(email)
    print(f"Email {i}: {'✓ consistent' if check['consistent'] else '✗ drift detected'}")




Email 1
{
  "sender_name": "Marcus Rivera",
  "sender_email": "m.rivera@northstar.io",
  "sender_company": "Northstar Logistics",
  "subject": "Interested in your enterprise plan",
  "is_reply": false,
  "mentioned_dates": [
    "2025-04-01"
  ],
  "mentioned_dollar_amounts": [
    85000.0
  ],
  "questions_asked": [
    "What's included in the enterprise tier?",
    "Can we schedule a demo next week?"
  ],
  "requested_actions": []
}

Email 2
{
  "sender_name": null,
  "sender_email": "jenny@example.com",
  "sender_company": null,
  "subject": "Fwd: Re: Fwd: pricing",
  "is_reply": true,
  "mentioned_dates": [
    "2023-12-15"
  ],
  "mentioned_dollar_amounts": [],
  "questions_asked": [],
  "requested_actions": [
    "Get back to the sender by 2023-12-15"
  ]
}

Email 3
{
  "sender_name": "David Okafor",
  "sender_email": "d.okafor@meridian-health.org",
  "sender_company": "Meridian Health",
  "subject": "Following up",
  "is_reply": false,
  "mentioned_dates": [],
  "mentioned_doll

In [7]:
from google.colab import files

output = [r.model_dump() for r in results]

with open("email_extractions.json", "w") as f:
    json.dump(output, f, indent=2)

files.download("email_extractions.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>